In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/data_spam_KKP_new.csv')
df['is_spam'] = df['is_spam'].map({True: 'SPAM', False: 'HAM'})
df = df.drop(columns=drop_cols)
df

,is_spam,post_text
0,HAM,Wakil Kepala BGN Nanik S. Deyang akan segera m...
1,HAM,Dimata mereka rakyat itu hanya angka & data..\...
2,HAM,Pakar Tegaskan Sekolah dan Orang Tua Bisa Meno...
3,HAM,Menu MBG hari ini ayam kentaki semoga anak2 se...
4,SPAM,"https://t.co/S8Mgl4DRDP, Jakarta - Presiden Pr..."
...,...,...
3291,SPAM,"Ep 3735b-Cyber Attack On EU Airports,[DS] 16 Y..."
3292,SPAM,OLIGARKING - MBG (MURDERED BY GOVERNMENT)\n#Mb...
3293,HAM,The real MBG in Chinna's School....no more dra...
3294,SPAM,Presiden Prabowo minta BGN rekrut koki terlati...


In [ ]:
df.to_csv('../data/data_spam_KKP_v3.csv', index=False)

In [6]:
# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

/Users/adsdigitalpartner/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/adsdigitalpartner/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
tokenizer = AutoTokenizer.from_pretrained("../models/v3-tuned")
model = AutoModelForSequenceClassification.from_pretrained("../models/v3-tuned")

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# ====== CONFIG ======
LABEL_NAMES = ["HAM", "SPAM"]  # urutan sesuai encoding model
BATCH_SIZE = 16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====== LOAD DATA CSV ======
df_test = pd.read_csv('../data/data_spam_KKP_v2.csv')

# Remove or fill missing values in post_text
texts = df_test["post_text"].fillna("").tolist()

# ====== TOKENISASI ======
encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

dataset = torch.utils.data.TensorDataset(
    encodings["input_ids"],
    encodings["attention_mask"]
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE)

# ====== PREDIKSI ======
model = model.to(device)
model.eval()
predictions_all = []
confidences_all = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)
        
        predictions_all.extend(preds)
        confidences_all.extend(probs.max(axis=-1))

# ====== SIMPAN HASIL ======
pred_labels_str = [LABEL_NAMES[p] for p in predictions_all]
df_test["predicted_label"] = pred_labels_str
df_test["confidence"] = confidences_all

df_test.to_excel("../data/hasil_prediksi_spam_KKP.xlsx", index=False)
print("\nHasil prediksi disimpan ke '../data/hasil_prediksi_spam_KKP.xlsx'")



Hasil prediksi disimpan ke '../data/hasil_prediksi_spam_KKP.xlsx'
